[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megusto0/rl-lab/blob/main/03_value_iteration.ipynb)


# 03. Value Iteration на FrozenLake

Цель ноутбука — решить FrozenLake-v1 методом динамического программирования при известной модели переходов.

**Результаты обучения:**
- извлекать модель MDP из Gymnasium;
- применять уравнение Беллмана оптимальности;
- получать жадную политику по функции ценности;
- тестировать найденную политику на эпизодах.

## Источник
Lapan M., *Deep Reinforcement Learning Hands-On*, глава 5; Sutton R. S., Barto A. G., глава 4.


In [ ]:
!pip install -q gymnasium


Подключим библиотеки и зададим общий seed. Для этого алгоритма PyTorch не требуется, но блок воспроизводимости одинаков во всех ноутбуках.


In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
except ImportError:
    pass


FrozenLake предоставляет полную модель переходов через `env.unwrapped.P`. Это позволяет выполнять планирование без обучающих взаимодействий со средой.


In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=True)
obs, info = env.reset(seed=SEED)
P = env.unwrapped.P
nS = env.observation_space.n
nA = env.action_space.n
print("states:", nS, "actions:", nA)
print("P[0][0]:", P[0][0])


Итеративно обновляем значения состояний до сходимости. В истории сохраняются номер итерации, максимальное изменение и сумма значений.


In [ ]:
GAMMA = 0.99
V = np.zeros(nS)
history = []
import time
t0 = time.perf_counter()

for i in range(1, 10000):
    V_new = np.zeros_like(V)
    for s in range(nS):
        V_new[s] = max(
            sum(p * (r + GAMMA * V[ns] * (not term))
                for p, ns, r, term in P[s][a])
            for a in range(nA))
    delta = np.max(np.abs(V_new - V))
    history.append((i, delta, V.sum()))
    V = V_new
    if delta < 1e-8:
        break
elapsed = time.perf_counter() - t0
print("converged in", i, "iterations")


Политика выбирает действие с максимальным ожидаемым возвратом. Для терминальных состояний формула корректно зануляет будущий вклад.


In [ ]:
pi = np.zeros(nS, dtype=int)
for s in range(nS):
    action_values = []
    for a in range(nA):
        value = sum(p * (r + GAMMA * V[ns] * (not term))
                    for p, ns, r, term in P[s][a])
        action_values.append(value)
    pi[s] = int(np.argmax(action_values))

print("policy:", pi.reshape(4, 4))


Проверим политику на 1000 эпизодах. Успех на FrozenLake определяется наградой 1 за достижение цели.


In [ ]:
test_rewards = []
for ep in range(1000):
    obs, info = env.reset(seed=SEED + ep)
    done, total_reward = False, 0.0
    while not done:
        obs, reward, terminated, truncated, info = env.step(int(pi[obs]))
        total_reward += reward
        done = terminated or truncated
    test_rewards.append(total_reward)

success_rate = np.mean(np.array(test_rewards) == 1.0)
mean_reward = np.mean(test_rewards)
print("success_rate:", success_rate)
print("mean_reward:", mean_reward)


Тепловая карта функции ценности и стрелки политики помогают проверить, что стратегия ведет к целевой клетке.


In [ ]:
os.makedirs("results", exist_ok=True)
arrows = np.array(["←", "↓", "→", "↑"])
plt.figure(figsize=(6, 5))
plt.imshow(V.reshape(4, 4), cmap="viridis")
for s, a in enumerate(pi):
    y, x = divmod(s, 4)
    plt.text(x, y, arrows[a], ha="center", va="center", color="white")
plt.colorbar(label="V(s)")
plt.title("Value function and greedy policy")
plt.xticks(range(4)); plt.yticks(range(4))
plt.savefig("03_value_policy.png", dpi=150, bbox_inches="tight")
plt.show()


Based on Lapan M., *Deep Reinforcement Learning Hands-On*, chapter 5.


In [ ]:
os.makedirs("results", exist_ok=True)
hist = pd.DataFrame(history, columns=["iteration", "max_delta", "V_sum"])
rows = []
for label, iteration in [("1", 1), ("10", 10), ("50", 50), ("100", 100)]:
    part = hist[hist["iteration"] == iteration]
    if len(part):
        r = part.iloc[0]
        rows.append((label, f"{r['max_delta']:.6g}", f"{r['V_sum']:.6f}"))
    else:
        rows.append((label, "-", "-"))
r = hist.iloc[-1]
rows.append(("До сходимости", f"{r['max_delta']:.6g}", f"{r['V_sum']:.6f}"))
df_history = pd.DataFrame(rows, columns=["Итерация", "max |ΔV|", "Сумма V(s) по всем s"])
df_summary = pd.DataFrame([
    ("Число итераций до сходимости", f"{int(hist['iteration'].iloc[-1])}"),
    ("Время вычисления, с", f"{elapsed:.4f}"),
    ("Доля успешных эпизодов", f"{100 * success_rate:.1f} %"),
    ("Средняя награда за эпизод", f"{mean_reward:.3f}"),
], columns=["Параметр", "Значение"])
df_results = df_history
print(df_history.to_string(index=False))
print(df_summary.to_string(index=False))
df_history.to_csv("results/03_value_iteration.csv", index=False)
df_summary.to_csv("results/03_value_iteration_summary.csv", index=False)
